# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")

30000 rows, 44 columns


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


I'm picking **Lane 4: CTR / Engagement Opportunity Scoring**. This isn't a random pick,it
comes directly out of something I found in Previous tasks. When I compared CTR by content type at the
same position tier comparison articles were consistently underperforming, even when I
controlled for rank. That told me position alone doesn't explain everything about CTR there's
a real, comparable gap sitting underneath it, and that's basically the whole premise of this
lane: find pages that are visible and well ranked but still under capturing clicks relative to
what similar pages get. It's a ranking/scoring problem ("which pages first?") not a
classification or clustering one and it produces something concrete a reviewer could actually
act on.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Question:** Among pages that are visible and reasonably well-positioned, which ones are
getting fewer clicks than they should be based on how similar pages (same position tier)
typically perform?

**Unit of analysis:** one page per row. Not a client, not a day a single content item a
reviewer could open and evaluate on its own.

**Output:** a ranked list of pages, each with a CTR-gap score (how far below its tier's typical
CTR it sits) and a short reason code explaining the flag something like "high impressions,
strong position, low CTR."

**Action someone takes:** a content or SEO reviewer uses the ranking to decide which pages to
open first for a title, meta description, or snippet rewrite.

**Who acts on it:** that reviewer, who has limited time and can't manually check every page
they need the list ordered so the most promising fixes are at the top.

**Cost of getting it wrong:** if the model flags a page that wasn't actually underperforming,
that's wasted reviewer time not a disaster, but a real cost when time is limited. If it misses
a page that genuinely needed fixing, that page just keeps quietly losing clicks it could've
been getting. Neither mistake is huge on its own, but they add up, which is why the ranking
needs to be backed by real comparisons instead of one blanket rule.

**Why this isn't just "train a model":** a flat rule like "flag anything under 1% CTR" sounds
simple, but it would just end up flagging every low-ranked page, since CTR naturally drops the
further down you go. That tells a reviewer nothing new. The actual question is comparative, is
this page's CTR low *for its position*, not low in general and that kind of "compared to what's
normal here" comparison is exactly where a model earns its place, not because models are
inherently smarter, but because this specific comparison is hard to hardcode cleanly.

**Putting it all together:** for a content reviewer deciding which pages to prioritize for a
rewrite, I'll build a ranked list from the starter data (and later the warehouse), scoring pages
by how far below their position tier's expected CTR they fall, and evaluate it with
precision@K on a held-out set. Getting it wrong costs either wasted reviewer time or a missed
quick win. A flat CTR threshold isn't enough because CTR is naturally tied to position. I'll only
claim results that are observed, directional and meant to support a decision not proof of
anything causal.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*



A few real numbers from the starter dataset that make me think this lane is worth digging into
for the next several weeks.

In [2]:
# How much does CTR shift just from position tier alone?
visible = df[df["impressions_90d"] >= 100]
ctr_by_tier = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Mean CTR by position tier:")
print(ctr_by_tier.round(4).to_string())
print(f"\nSpread: {ctr_by_tier.max() - ctr_by_tier.min():.4f} — this alone shows why a flat "
      f"CTR cutoff would be misleading unless you compare pages within the same tier.")

Mean CTR by position tier:
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554

Spread: 0.2993 — this alone shows why a flat CTR cutoff would be misleading unless you compare pages within the same tier.


In [3]:
# Within the same tier, does content_type still create a gap?
strong_visibility = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)]
gap = strong_visibility.groupby("content_type")["ctr"].mean().sort_values()
print("Mean CTR by content_type, restricted to strong-visibility pages (position 1-20):")
print(gap.round(4).to_string())
print(f"\nGap between the lowest and highest content_type: {gap.max() - gap.min():.4f}")

Mean CTR by content_type, restricted to strong-visibility pages (position 1-20):
content_type
comparison article    0.0879
keyword article       0.3117
feedly article        0.4671

Gap between the lowest and highest content_type: 0.3792


In [4]:
# How many pages would actually qualify as review candidates under a reasonable rule?
candidates = df[
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) & (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
]
print(f"{len(candidates):,} pages ({len(candidates)/len(df):.1%} of the dataset) meet the "
      f"'visible + well-positioned + low CTR' criteria — a real, sizeable pool to work with.")

9,759 pages (32.5% of the dataset) meet the 'visible + well-positioned + low CTR' criteria — a real, sizeable pool to work with.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*



**What I can say:** which pages, in this anonymized sample, show a CTR gap compared to similar
pages at the same position tier. That's an observed, directional pattern — not a guarantee. What
this produces is a decision-support ranking meant to help a reviewer spend limited time more
wisely, with reason codes attached so the ranking is inspectable, not a black box.

**What I can't say:** that rewriting a title or meta description will actually *cause* CTR to go
up — proving that would need a real experiment, like an A/B test on a live page, which this
dataset can't give me. I also can't make any claim about how Google's ranking algorithm works.
And since this is observational data, any gap I find between content types could be explained by
something else I haven't controlled for — topic, intent, audience — so "content type causes low
CTR" isn't something this data can actually prove.

## Self-check

Before you submit, confirm each line honestly:

- [done ] Every section above is filled — markdown thinking AND the code that backs it
- [done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ done] No client names, URLs, or private queries anywhere
- [ done] My claims use careful words: observed, measured, directional, decision-support
- [ done] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.